# Banking System — Demos (Days 1–6)

This notebook duplicates the demonstrations from `src/main.py` and `src/simulation.py`, one section per day. Run the cells top to bottom.

> Run Jupyter from the repository root so `from src.domain import ...` resolves.

In [ ]:
from datetime import datetime
from decimal import Decimal

from src.domain import (
    Bank, Client, Currency, AccountStatus, AssetClass,
    BankAccount, SavingsAccount, PremiumAccount, InvestmentAccount,
    Transaction, TransactionType, TransactionPriority,
    TransactionQueue, TransactionProcessor,
    RiskAnalyzer, AuditReporter, DomainError, UnderageError,
)

## Day 1 — Base bank account model
Abstract account + `BankAccount`: validation, statuses (active/frozen/closed), multi-currency, custom errors.

In [ ]:
active = BankAccount("Alice", balance=1000, currency=Currency.USD)
frozen = BankAccount("Bob", balance=500, status=AccountStatus.FROZEN, currency=Currency.EUR)
print(active)
print(frozen)

# Operations on a frozen account are blocked.
for label, op in (("deposit", lambda: frozen.deposit(100)),
                  ("withdraw", lambda: frozen.withdraw(100))):
    try:
        op()
    except DomainError as exc:
        print(f"{label} blocked -> {type(exc).__name__}: {exc}")

# Valid operations on the active account.
active.deposit(250)
active.withdraw(400)
print(active)
print(active.get_account_info())

## Day 2 — Advanced account types
`SavingsAccount`, `PremiumAccount`, `InvestmentAccount` with polymorphic `withdraw` / `get_account_info` / `__str__`.

In [ ]:
savings = SavingsAccount("Carol", balance=1000, min_balance=200,
                         monthly_rate="0.05", currency=Currency.RUB)
print(savings)
print("interest:", savings.apply_monthly_interest(), "-> balance", savings.balance)
try:
    savings.withdraw(950)
except DomainError as exc:
    print("withdraw(950) blocked ->", type(exc).__name__, exc)
print("withdraw(300) ok -> balance", savings.withdraw(300))

premium = PremiumAccount("Dave", balance=100, overdraft_limit=500,
                         transaction_fee=5, currency=Currency.USD)
print(premium)
print("withdraw(400) with overdraft -> balance", premium.withdraw(400))

invest = InvestmentAccount("Eve", balance=10000, currency=Currency.EUR)
invest.invest(AssetClass.STOCKS, 4000)
invest.invest(AssetClass.BONDS, 3000)
invest.invest(AssetClass.ETF, 2000)
print(invest)
print("projected yearly growth:", invest.project_yearly_growth(), invest.currency.value)

## Day 3 — Bank system & client management
`Client` (age check, PIN) and `Bank` (accounts, authentication, security, analytics).

In [ ]:
bank = Bank("Demo Bank")

try:
    Client("Young Tom", 16, pin="0000")
except UnderageError as exc:
    print("underage rejected ->", exc)

alice = bank.add_client(Client("Alice Smith", 30, pin="1234", client_id="ALICE001"))
bob = bank.add_client(Client("Bob Jones", 45, pin="4321", client_id="BOB002"))
a1 = bank.open_account(alice.client_id, "savings", balance=5000, min_balance=500,
                       monthly_rate="0.04", currency=Currency.USD)
bank.open_account(alice.client_id, "premium", balance=2000, overdraft_limit=1000,
                  transaction_fee=10, currency=Currency.USD)
bank.open_account(bob.client_id, "investment", balance=8000, currency=Currency.EUR)
print(alice, "| accounts:", alice.account_numbers)

# Three wrong PINs block the client.
for _ in range(3):
    bank.authenticate_client("BOB002", "0000")
print("Bob blocked:", bob.is_blocked, "| flags:", bob.suspicious_flags)

print("Total balance per currency:", bank.get_total_balance())
print("Clients ranking:", bank.get_clients_ranking())

## Day 4 — Transaction system
`Transaction`, `TransactionQueue` (priority/deferred/cancel), `TransactionProcessor` (fees, conversion, retries).

In [ ]:
bank = Bank("Demo Bank")
bank.add_client(Client("Alice", 30, pin="1", client_id="C1"))
bank.add_client(Client("Bob", 30, pin="2", client_id="C2"))
usd = bank.open_account("C1", "bank", balance=1000, currency=Currency.USD)
eur = bank.open_account("C2", "bank", balance=1000, currency=Currency.EUR)

processor = TransactionProcessor(bank)
queue = TransactionQueue()
transactions = [
    Transaction(TransactionType.DEPOSIT, 200, Currency.USD, receiver=usd.account_id),
    Transaction(TransactionType.WITHDRAWAL, 150, Currency.USD, sender=usd.account_id),
    Transaction(TransactionType.TRANSFER, 100, Currency.USD,
                sender=usd.account_id, receiver=eur.account_id),
    Transaction(TransactionType.EXTERNAL_TRANSFER, 100, Currency.USD,
                sender=usd.account_id, receiver="EXT-OUT"),
    Transaction(TransactionType.WITHDRAWAL, 999999, Currency.USD, sender=usd.account_id),
    Transaction(TransactionType.DEPOSIT, 25, Currency.USD, receiver=usd.account_id,
                priority=TransactionPriority.HIGH),
    Transaction(TransactionType.DEPOSIT, 5, Currency.USD, receiver=usd.account_id,
                transaction_id="CANCELLED-1"),
]
for tx in transactions:
    queue.add(tx)
queue.cancel("CANCELLED-1")

for tx in processor.process_queue(queue):
    suffix = f"  ({tx.failure_reason})" if tx.failure_reason else ""
    print(tx, suffix)
print("Final balances: USD", usd.balance, "EUR", eur.balance)

## Day 5 — Audit & risk analysis
`RiskAnalyzer` flags suspicious operations; the processor blocks high-risk ones; `AuditReporter` builds reports.

In [ ]:
midday = datetime(2026, 6, 9, 12, 0, 0)
night = datetime(2026, 6, 9, 2, 0, 0)
bank = Bank("Demo Bank", now=lambda: midday)
bank.add_client(Client("Alice", 30, pin="1", client_id="C1"))
account = bank.open_account("C1", "bank", balance=100000, currency=Currency.USD)

analyzer = RiskAnalyzer(large_amount=10000, frequency_limit=3)
processor = TransactionProcessor(bank, risk=analyzer, now=lambda: night)
ops = [
    Transaction(TransactionType.WITHDRAWAL, 100, Currency.USD, sender=account.account_id),
    Transaction(TransactionType.WITHDRAWAL, 200, Currency.USD, sender=account.account_id),
    Transaction(TransactionType.EXTERNAL_TRANSFER, 50000, Currency.USD,
                sender=account.account_id, receiver="EXT-NEW"),
    Transaction(TransactionType.WITHDRAWAL, 50, Currency.USD, sender=account.account_id),
    Transaction(TransactionType.WITHDRAWAL, 60, Currency.USD, sender=account.account_id),
]
for tx in ops:
    processor.process(tx)
    print(tx, ("-> " + tx.failure_reason) if tx.failure_reason else "")

reporter = AuditReporter(processor.audit)
print("\nSuspicious operations:", len(reporter.suspicious_operations()))
print("Client risk profile:", reporter.client_risk_profile(bank, "C1"))
print("Error statistics:", reporter.error_statistics())

## Day 6 — Comprehensive simulation
The full capstone: a bank with 7 clients and 12 accounts, ~40 transactions (some erroneous, some suspicious), user scenarios, and reports.

In [ ]:
from src.simulation import run
run()